# Build Your Own Mini GPT — Beginner-Friendly LLM Notebook

This notebook contains everything you need to build, train, and experiment with your own **mini GPT-style language model** from the ground up.

The goal of this notebook is not simply to provide a working LLM, but to help you **understand how an LLM is built and how you should approach the process step by step**.

It includes the complete workflow, from preparing the dataset to implementing the Transformer architecture, attention mechanisms, model training, and testing the model's output. Each major component also includes test code directly below its implementation, allowing you to see what each part is doing and verify that it works before moving forward.

The notebook is intentionally kept simple and beginner-friendly, making it a practical starting point for anyone interested in learning how GPT-style models work.

### What's Included

* Dataset preparation and processing
* Tokenization and data loading
* Transformer architecture
* Attention mechanism
* Model implementation
* Training pipeline
* Model evaluation and testing
* Output generation
* Tests and examples for individual components
* Complete end-to-end training workflow

### Important Notes

* **The notebook is currently in its raw form** and has not been heavily edited or polished. It is intentionally provided this way so you can explore the implementation and understand the development process.
* **Each code section is followed by its corresponding test/output**, so you can observe what happens at every stage.
* **The model contains approximately 5.2 million parameters**, making it small enough to experiment with while still demonstrating the core concepts behind GPT-style models.
* The notebook uses **existing libraries** for certain components to keep the implementation accessible and easier to understand for beginners.
* If you use a **different dataset**, you may need to modify some parts of the preprocessing and training code depending on the dataset format.
* This github also has the file you need to make this.

### The Bigger Goal

This notebook is designed to teach more than just *how to make an LLM*.

It is also about understanding **how to approach building one** — how the individual components fit together, how to test each stage, and how the complete system evolves from data into a working language model.

For beginners, this should provide a solid foundation for understanding the fundamentals of GPT-style architectures.

And this is only the beginning.

I will also be working on **more modern and advanced architectures where the major components are implemented from scratch without relying heavily on existing high-level libraries**.

If you're interested in understanding what's happening inside modern LLMs rather than simply using them, stay tuned.


In [1]:
!pip install -q transformers datasets tokenizers accelerate safetensors huggingface_hub sentencepiece

In [2]:
import torch
import transformers
import datasets
import tokenizers

print(torch.__version__)
print(transformers.__version__)
print(datasets.__version__)
print(tokenizers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cpu
5.0.0
5.0.0
0.22.2
False


AssertionError: Torch not compiled with CUDA enabled

In [3]:
import os

PROJECT = "MiniGPT"

folders = [
    f"{PROJECT}",
    f"{PROJECT}/src",
    f"{PROJECT}/tokenizer",
    f"{PROJECT}/data",
    f"{PROJECT}/models",
    f"{PROJECT}/checkpoints",
    f"{PROJECT}/outputs",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure created!")

Project structure created!


# Tokenizer

In [8]:
%%writefile MiniGPT/train_tokenizer.py
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast
import os

# ============================
# Configuration
# ============================

MODEL_NAME = "MiniGPT"

DATASET_NAME = "roneneldan/TinyStories"

VOCAB_SIZE = 20000

SPECIAL_TOKENS = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>",
]

OUTPUT_DIR = f"{MODEL_NAME}/tokenizer"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================
# Load Dataset
# ============================

print("Loading TinyStories...")

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

print(dataset)

# ============================
# Generator
# ============================

def get_training_corpus():

    for sample in dataset:

        yield sample["text"]

# ============================
# Build Tokenizer
# ============================

tokenizer = Tokenizer(BPE(unk_token="<unk>"))

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
)

print("Training tokenizer...")

tokenizer.train_from_iterator(
    get_training_corpus(),
    trainer=trainer,
)

# ============================
# Wrap for Hugging Face
# ============================

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>",
)

hf_tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Tokenizer saved to {OUTPUT_DIR}")

Overwriting MiniGPT/train_tokenizer.py


# training of tokenizer

In [9]:
!python MiniGPT/train_tokenizer.py

Loading TinyStories...
Dataset({
    features: ['text'],
    num_rows: 2119719
})
Training tokenizer...
[00:00:00] Tokenize words                 ██████████████████ 60513    /    60513[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 60513    /    60513
[00:00:00] Compute merges                 ██████████████████ 19826    /    19826
Tokenizer saved to MiniGPT/tokenizer


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("MiniGPT/tokenizer")

print(tokenizer)
print()
print("Vocabulary Size:", tokenizer.vocab_size)
print("Special Tokens:", tokenizer.special_tokens_map)

TokenizersBackend(name_or_path='MiniGPT/tokenizer', vocab_size=20000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

Vocabulary Size: 20000
Special Tokens: {'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>'}


In [11]:
text = "Hello! My name is MiniGPT."

tokens = tokenizer.encode(text)

print(tokens)

[1178, 4, 1672, 1095, 193, 6411, 75, 42, 51, 55, 17]


In [12]:
decoded = tokenizer.decode(tokens)

print(decoded)

Hello ! My name is Min i G P T .


In [ ]:
text = "Hello! My name is MiniGPT"

tokens = tokenizer.encode(text)

print(tokens)
print(tokenizer.decode(tokens))

# **config file created**

In [ ]:
%%writefile MiniGPT/src/config.py

from dataclasses import dataclass

@dataclass
class MiniGPTConfig:
    # Tokenizer
    vocab_size: int = 8000
    tokenizer_path: str = "/kaggle/working/MiniGPT/tokenizer"

    # Model
    context_length: int = 128
    embedding_dim: int = 256
    num_heads: int = 4
    num_layers: int = 4
    dropout: float = 0.1
    head_dim = embedding_dim // num_heads

    # Feed Forward Network
    ffn_dim: int = 1024

    # Training
    batch_size: int = 16
    learning_rate: float = 3e-4
    epochs: int = 3

    assert embedding_dim % num_heads == 0

    # Special Tokens
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2
    unk_token_id: int = 3

# Updated config 

In [13]:
%%writefile MiniGPT/src/config.py
from dataclasses import dataclass


@dataclass
class MiniGPTConfig:
    # Tokenizer
    vocab_size: int = 20000
    tokenizer_path: str = "/kaggle/working/MiniGPT/tokenizer"

    # Model
    context_length: int = 128
    embedding_dim: int = 256
    num_heads: int = 4
    num_layers: int = 4
    dropout: float = 0.1

    # Feed Forward Network
    ffn_dim: int = 1024

    # Training
    batch_size: int = 16
    learning_rate: float = 3e-4
    epochs: int = 20

    # Special Tokens
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2
    unk_token_id: int = 3

    def __post_init__(self):
        assert self.embedding_dim % self.num_heads == 0, \
            "embedding_dim must be divisible by num_heads"

        self.head_dim = self.embedding_dim // self.num_heads

Writing MiniGPT/src/config.py


# Creating dataset 

 roneneldan/TinyStories : purpose of using this is only its a good dataset of stories so, everyone can able to understand the output clearly.

In [ ]:
%%writefile MiniGPT/src/dataset.py

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

from src.config import MiniGPTConfig


class TinyStoriesDataset(Dataset):

    def __init__(
        self,
        tokenizer_path,
        max_stories=50000,
    ):

        self.config = MiniGPTConfig()

        print("Loading tokenizer...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_path
        )

        print("Loading TinyStories...")

        dataset = load_dataset(
            "roneneldan/TinyStories",
            split="train"
        )

        dataset = dataset.select(
            range(max_stories)
        )

        print(f"Loaded {len(dataset)} stories")

        print("Tokenizing stories...")

        all_tokens = []

        for sample in dataset:

            tokens = self.tokenizer.encode(
                sample["text"],
                add_special_tokens=False
            )

            tokens.append(
                self.config.eos_token_id
            )

            all_tokens.extend(tokens)

        print("Total tokens:", len(all_tokens))

        self.tokens = all_tokens

    def __len__(self):

        return (
            len(self.tokens) - 1
        ) // self.config.context_length

    def __getitem__(self, idx):

        start = idx * self.config.context_length

        end = start + self.config.context_length + 1

        chunk = self.tokens[start:end]

        input_ids = torch.tensor(
            chunk[:-1],
            dtype=torch.long
        )

        labels = torch.tensor(
            chunk[1:],
            dtype=torch.long
        )

        return input_ids, labels

# Test dataset code and run test 

In [ ]:
%%writefile MiniGPT/test_dataset.py

from src.dataset import TinyStoriesDataset

dataset = TinyStoriesDataset(
    tokenizer_path="MiniGPT/tokenizer",
    max_stories=100
)

print()

print("Dataset Size:", len(dataset))

print()

x, y = dataset[0]

print("Input Shape :", x.shape)
print("Label Shape :", y.shape)

print()

print(x)

print()

print(y)

In [ ]:
!python MiniGPT/test_dataset.py

# model and model code test 

In [ ]:
%%writefile MiniGPT/src/model.py
import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class TokenEmbedding(nn.Module):
    """
    Converts token IDs into learnable vectors.
    """

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.embedding_dim,
        )

    def forward(self, input_ids):

        return self.embedding(input_ids)

In [ ]:
%%writefile MiniGPT/test_embedding.py
import torch

from src.config import MiniGPTConfig
from src.model import TokenEmbedding

config = MiniGPTConfig()

embedding = TokenEmbedding(config)

input_ids = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(2, 8)
)

print("Input IDs")
print(input_ids)

print()

output = embedding(input_ids)

print("Output Shape:", output.shape)

print()

print(output)

# 2 → number of sequences in the batch.
# 4 → number of tokens in each sequence.
# 256 → embedding vector for each token.

B = Batch Size
T = Number of Tokens (Context Length)
C = Embedding Dimension (Channels)

In [ ]:
!python MiniGPT/test_embedding.py

# Adding postion embedding to model now 

In [ ]:
%%writefile MiniGPT/src/model.py

import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class TokenEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.embedding = nn.Embedding(
            config.vocab_size,
            config.embedding_dim
        )

    def forward(self, input_ids):
        return self.embedding(input_ids)


class PositionalEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.embedding = nn.Embedding(
            config.context_length,
            config.embedding_dim
        )

    def forward(self, input_ids):

        batch_size, seq_length = input_ids.shape

        positions = torch.arange(
            seq_length,
            device=input_ids.device
        )

        positions = positions.unsqueeze(0).expand(
            batch_size,
            seq_length
        )

        return self.embedding(positions)


class GPTEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.token_embedding = TokenEmbedding(config)
        self.position_embedding = PositionalEmbedding(config)

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):

        token_embeddings = self.token_embedding(input_ids)

        position_embeddings = self.position_embedding(input_ids)

        embeddings = token_embeddings + position_embeddings

        return self.dropout(embeddings)

In [ ]:
%%writefile MiniGPT/test_gpt_embedding.py

import torch

from src.config import MiniGPTConfig
from src.model import GPTEmbedding

config = MiniGPTConfig()

embedding = GPTEmbedding(config)

input_ids = torch.randint(
    0,
    config.vocab_size,
    (2, 8)
)

print("Input Shape:", input_ids.shape)

output = embedding(input_ids)

print("Output Shape:", output.shape)

print(output)

In [ ]:
!python MiniGPT/test_gpt_embedding.py

# Attention layer and test 

In [ ]:
%%writefile MiniGPT/src/attention.py

import math
import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class SelfAttention(nn.Module):

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding_dim = config.embedding_dim

        # Query projection
        self.query = nn.Linear(
            config.embedding_dim,
            config.embedding_dim,
            bias=False
        )

        # Key projection
        self.key = nn.Linear(
            config.embedding_dim,
            config.embedding_dim,
            bias=False
        )

        # Value projection
        self.value = nn.Linear(
            config.embedding_dim,
            config.embedding_dim,
            bias=False
        )

    def forward(self, x):

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        return q, k, v

In [ ]:
%%writefile  MiniGPT/test_attention.py
# MiniGPT/test_attention.py

import torch

from src.config import MiniGPTConfig
from src.attention import SelfAttention

config = MiniGPTConfig()

attention = SelfAttention(config)

x = torch.randn(2, 8, config.embedding_dim)

q, k, v = attention(x)

print("Input :", x.shape)
print("Query :", q.shape)
print("Key   :", k.shape)
print("Value :", v.shape)

In [ ]:
!python  MiniGPT/test_attention.py

# Attention layer with better forword pass and the test run 

In [ ]:
%%writefile MiniGPT/src/attention.py

import math
import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class SelfAttention(nn.Module):

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding_dim = config.embedding_dim

        # Query projection
        self.query = nn.Linear(
            config.embedding_dim,
            config.embedding_dim,
            bias=False
        )

        # Key projection
        self.key = nn.Linear(
            config.embedding_dim,
            config.embedding_dim,
            bias=False
        )

        # Value projection
        self.value = nn.Linear(
            config.embedding_dim,
            config.embedding_dim,
            bias=False
        )

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.embedding_dim)

        seq_length = x.size(1)
        mask = torch.triu(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device
            ),
            diagonal=1
        )

        scores = scores.masked_fill(
            mask == 1,
            float("-inf")
        )

        attention = torch.softmax(
            scores,
            dim=-1
        )

        output = attention @ v
        return output

In [ ]:
%%writefile  MiniGPT/test_attention.py

import torch

from src.config import MiniGPTConfig
from src.attention import SelfAttention

config = MiniGPTConfig()

model = SelfAttention(config)

x = torch.randn(
    2,
    8,
    config.embedding_dim
)

output = model(x)

print("Input :", x.shape)
print("Output:", output.shape)

In [ ]:
!python  MiniGPT/test_attention.py

# 4 head implantation  which have 256 dimension and every head has 256/4 = 64 dimensions
> where every head know about the token like grammer pronoun postion etc 

In [ ]:
%%writefile MiniGPT/test_reshape.py
import torch

x = torch.randn(2, 8, 256)

print("Original :", x.shape)

x = x.view(2, 8, 4, 64)

print("View :", x.shape)

x = x.transpose(1, 2)

print("Transpose :", x.shape)

In [ ]:
!python MiniGPT/test_reshape.py

# attention layer for 4 head implantation 

In [14]:
%%writefile MiniGPT/src/attention.py

import math
import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class MultiHeadAttention(nn.Module):

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding_dim = config.embedding_dim
        self.num_heads = config.num_heads
        self.head_dim = config.head_dim

        assert self.embedding_dim == self.num_heads * self.head_dim

        # Q, K, V projections
        self.query = nn.Linear(
            self.embedding_dim,
            self.embedding_dim,
            bias=False
        )

        self.key = nn.Linear(
            self.embedding_dim,
            self.embedding_dim,
            bias=False
        )

        self.value = nn.Linear(
            self.embedding_dim,
            self.embedding_dim,
            bias=False
        )

        # Output Projection (Wo)
        self.out_proj = nn.Linear(
            self.embedding_dim,
            self.embedding_dim
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):

        batch_size, seq_length, _ = x.shape

        # Step 1
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

Writing MiniGPT/src/attention.py


In [15]:
%%writefile MiniGPT/test_multihead.py

import torch

from src.config import MiniGPTConfig
from src.attention import MultiHeadAttention

config = MiniGPTConfig()

model = MultiHeadAttention(config)

x = torch.randn(
    2,
    8,
    config.embedding_dim
)

batch_size, seq_length, _ = x.shape

q = model.query(x)

print("Original:", q.shape)

q = q.view(
    batch_size,
    seq_length,
    model.num_heads,
    model.head_dim
)

print("View:", q.shape)

q = q.transpose(1, 2)

print("Transpose:", q.shape)

Writing MiniGPT/test_multihead.py


In [16]:
!python MiniGPT/test_multihead.py

Original: torch.Size([2, 8, 256])
View: torch.Size([2, 8, 4, 64])
Transpose: torch.Size([2, 4, 8, 64])


# final attention.py for everything like scoring and all and  testing the code below

In [17]:
%%writefile MiniGPT/src/attention.py

import math
import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Causal Self-Attention
    """

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding_dim = config.embedding_dim
        self.num_heads = config.num_heads
        self.head_dim = config.head_dim

        assert self.embedding_dim % self.num_heads == 0

        # Query, Key, Value projections
        self.query = nn.Linear(
            self.embedding_dim,
            self.embedding_dim,
            bias=False
        )

        self.key = nn.Linear(
            self.embedding_dim,
            self.embedding_dim,
            bias=False
        )

        self.value = nn.Linear(
            self.embedding_dim,
            self.embedding_dim,
            bias=False
        )

        # Output projection (Wo)
        self.out_proj = nn.Linear(
            self.embedding_dim,
            self.embedding_dim
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):

        batch_size, seq_length, _ = x.shape

        # -------------------------------------------------
        # Project input into Query, Key and Value
        # -------------------------------------------------
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # -------------------------------------------------
        # Split into multiple heads
        # (B, T, C) -> (B, T, H, D)
        # -------------------------------------------------
        q = q.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        k = k.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        v = v.view(
            batch_size,
            seq_length,
            self.num_heads,
            self.head_dim
        )

        # -------------------------------------------------
        # (B, T, H, D) -> (B, H, T, D)
        # -------------------------------------------------
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # -------------------------------------------------
        # Scaled Dot Product Attention
        # -------------------------------------------------
        scores = q @ k.transpose(-2, -1)

        scores = scores / math.sqrt(self.head_dim)

        # -------------------------------------------------
        # Causal Mask
        # -------------------------------------------------
        mask = torch.triu(
            torch.ones(
                seq_length,
                seq_length,
                device=x.device
            ),
            diagonal=1
        )

        scores = scores.masked_fill(
            mask == 1,
            float("-inf")
        )

        # -------------------------------------------------
        # Softmax
        # -------------------------------------------------
        attention = torch.softmax(
            scores,
            dim=-1
        )

        attention = self.dropout(attention)

        # -------------------------------------------------
        # Weighted sum of Values
        # -------------------------------------------------
        output = attention @ v

        # -------------------------------------------------
        # Combine Heads
        # (B, H, T, D) -> (B, T, H, D)
        # -------------------------------------------------
        output = output.transpose(1, 2)

        # -------------------------------------------------
        # (B, T, H, D) -> (B, T, C)
        # -------------------------------------------------
        output = output.contiguous().view(
            batch_size,
            seq_length,
            self.embedding_dim
        )

        # -------------------------------------------------
        # Final Linear Projection
        # -------------------------------------------------
        output = self.out_proj(output)

        output = self.dropout(output)

        return output

Overwriting MiniGPT/src/attention.py


In [18]:
%%writefile MiniGPT/test_multihead.py

import torch

from src.config import MiniGPTConfig
from src.attention import MultiHeadAttention

config = MiniGPTConfig()

model = MultiHeadAttention(config)

x = torch.randn(
    2,
    8,
    config.embedding_dim
)

output = model(x)

print("Input Shape :", x.shape)
print("Output Shape:", output.shape)

Overwriting MiniGPT/test_multihead.py


In [19]:
!python MiniGPT/test_multihead.py

Input Shape : torch.Size([2, 8, 256])
Output Shape: torch.Size([2, 8, 256])


# Feed Forward Network second brain of llm after attention head 
> it thinks after attention head done with info gathering 

In [20]:
%%writefile MiniGPT/src/mlp.py

import torch.nn as nn

from src.config import MiniGPTConfig


class FeedForward(nn.Module):
    """
    Feed Forward Network (MLP)
    """

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                config.embedding_dim,
                config.ffn_dim
            ),

            nn.GELU(),

            nn.Dropout(config.dropout),

            nn.Linear(
                config.ffn_dim,
                config.embedding_dim
            ),

            nn.Dropout(config.dropout)

        )

    def forward(self, x):
        return self.net(x)

Writing MiniGPT/src/mlp.py


In [21]:
%%writefile  MiniGPT/test_mlp.py
import torch

from src.config import MiniGPTConfig
from src.mlp import FeedForward

config = MiniGPTConfig()

mlp = FeedForward(config)

x = torch.randn(
    2,
    8,
    config.embedding_dim
)

output = mlp(x)

print("Input :", x.shape)
print("Output:", output.shape)

Writing MiniGPT/test_mlp.py


In [22]:
!python MiniGPT/test_mlp.py

Input : torch.Size([2, 8, 256])
Output: torch.Size([2, 8, 256])


# Now the transformer part 

In [26]:
%%writefile MiniGPT/src/transformer.py

import torch.nn as nn

from src.attention import MultiHeadAttention
from src.mlp import FeedForward
from src.config import MiniGPTConfig


class TransformerBlock(nn.Module):

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.ln1 = nn.LayerNorm(config.embedding_dim)
        self.attention = MultiHeadAttention(config)

        self.ln2 = nn.LayerNorm(config.embedding_dim)
        self.mlp = FeedForward(config)

    def forward(self, x):

        # Attention + Residual
        x = x + self.attention(self.ln1(x))

        # MLP + Residual
        x = x + self.mlp(self.ln2(x))

        return x

Overwriting MiniGPT/src/transformer.py


In [27]:
%%writefile MiniGPT/test_transformer.py
import torch

from src.config import MiniGPTConfig
from src.transformer import TransformerBlock

config = MiniGPTConfig()

block = TransformerBlock(config)

x = torch.randn(
    2,
    8,
    config.embedding_dim
)

output = block(x)

print("Input :", x.shape)
print("Output:", output.shape)

Writing MiniGPT/test_transformer.py


In [28]:
!python MiniGPT/test_transformer.py

Input : torch.Size([2, 8, 256])
Output: torch.Size([2, 8, 256])


# creating the model.py again for updates and creating separate embedding 

In [32]:
%%writefile MiniGPT/src/model.py

import torch
import torch.nn as nn

from src.config import MiniGPTConfig
from src.embeddings import GPTEmbedding
from src.transformer import TransformerBlock


class MiniGPT(nn.Module):
    """
    MiniGPT Language Model
    """

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding = GPTEmbedding(config)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config)
                for _ in range(config.num_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(
            config.embedding_dim
        )

        self.lm_head = nn.Linear(
            config.embedding_dim,
            config.vocab_size,
            bias=False
        )
        
        # Weight Tying (GPT-2 Style)  this is after the training is added 
        self.lm_head.weight = self.embedding.token_embedding.weight

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

Overwriting MiniGPT/src/model.py


In [33]:
%%writefile MiniGPT/src/embeddings.py

import torch
import torch.nn as nn

from src.config import MiniGPTConfig


class PositionalEmbedding(nn.Module):

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.embedding = nn.Embedding(
            config.context_length,
            config.embedding_dim
        )

    def forward(self, x):
        batch_size, seq_length = x.shape

        positions = torch.arange(
            seq_length,
            device=x.device
        ).unsqueeze(0)

        return self.embedding(positions)


class GPTEmbedding(nn.Module):

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.embedding_dim
        )

        self.position_embedding = PositionalEmbedding(config)

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_ids):

        token_embeddings = self.token_embedding(input_ids)

        position_embeddings = self.position_embedding(input_ids)

        embeddings = token_embeddings + position_embeddings

        return self.dropout(embeddings)

Writing MiniGPT/src/embeddings.py


In [34]:
%%writefile MiniGPT/test_model.py

import torch

from src.config import MiniGPTConfig
from src.model import MiniGPT

config = MiniGPTConfig()

model = MiniGPT(config)

x = torch.randint(
    0,
    config.vocab_size,
    (2, 8)
)

output = model(x)

print("Input :", x.shape)
print("Output:", output.shape)

Writing MiniGPT/test_model.py


In [35]:
!python MiniGPT/test_model.py

Input : torch.Size([2, 8])
Output: torch.Size([2, 8, 20000])


# training the model 

In [36]:
%%writefile MiniGPT/src/trainer.py

import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

from src.config import MiniGPTConfig


class Trainer:

    def __init__(self, model, train_loader, config: MiniGPTConfig):

        self.model = model
        self.train_loader = train_loader
        self.config = config

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.model.to(self.device)

        self.criterion = nn.CrossEntropyLoss()

        self.optimizer = AdamW(
            self.model.parameters(),
            lr=config.learning_rate
        )

    def train_epoch(self):

        self.model.train()

        total_loss = 0

        progress_bar = tqdm(
            self.train_loader,
            desc="Training",
            leave=False
        )

        for input_ids, labels in progress_bar:

            input_ids = input_ids.to(self.device)
            labels = labels.to(self.device)

            self.optimizer.zero_grad()

            logits = self.model(input_ids)

            loss = self.criterion(
                logits.view(-1, self.config.vocab_size),
                labels.view(-1)
            )

            loss.backward()

            self.optimizer.step()

            total_loss += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        return total_loss / len(self.train_loader)

Writing MiniGPT/src/trainer.py


# checking every-component in mini-gpt 

In [38]:
import sys

sys.path.append("/kaggle/working/MiniGPT")
from src.config import MiniGPTConfig
from src.model import MiniGPT

config = MiniGPTConfig()

model = MiniGPT(config)

In [39]:
for name, param in model.named_parameters():
    print(name, param.shape)

embedding.token_embedding.weight torch.Size([20000, 256])
embedding.position_embedding.embedding.weight torch.Size([128, 256])
blocks.0.ln1.weight torch.Size([256])
blocks.0.ln1.bias torch.Size([256])
blocks.0.attention.query.weight torch.Size([256, 256])
blocks.0.attention.key.weight torch.Size([256, 256])
blocks.0.attention.value.weight torch.Size([256, 256])
blocks.0.attention.out_proj.weight torch.Size([256, 256])
blocks.0.attention.out_proj.bias torch.Size([256])
blocks.0.ln2.weight torch.Size([256])
blocks.0.ln2.bias torch.Size([256])
blocks.0.mlp.net.0.weight torch.Size([1024, 256])
blocks.0.mlp.net.0.bias torch.Size([1024])
blocks.0.mlp.net.3.weight torch.Size([256, 1024])
blocks.0.mlp.net.3.bias torch.Size([256])
blocks.1.ln1.weight torch.Size([256])
blocks.1.ln1.bias torch.Size([256])
blocks.1.attention.query.weight torch.Size([256, 256])
blocks.1.attention.key.weight torch.Size([256, 256])
blocks.1.attention.value.weight torch.Size([256, 256])
blocks.1.attention.out_proj.wei

In [40]:
print("=" * 80)
print("MiniGPT Parameters")
print("=" * 80)

total_params = 0

for name, param in model.named_parameters():
    print(f"{name:50} {str(param.shape):25} Trainable={param.requires_grad}")
    total_params += param.numel()

print("=" * 80)
print(f"Total Parameters: {total_params:,}")

MiniGPT Parameters
embedding.token_embedding.weight                   torch.Size([20000, 256])  Trainable=True
embedding.position_embedding.embedding.weight      torch.Size([128, 256])    Trainable=True
blocks.0.ln1.weight                                torch.Size([256])         Trainable=True
blocks.0.ln1.bias                                  torch.Size([256])         Trainable=True
blocks.0.attention.query.weight                    torch.Size([256, 256])    Trainable=True
blocks.0.attention.key.weight                      torch.Size([256, 256])    Trainable=True
blocks.0.attention.value.weight                    torch.Size([256, 256])    Trainable=True
blocks.0.attention.out_proj.weight                 torch.Size([256, 256])    Trainable=True
blocks.0.attention.out_proj.bias                   torch.Size([256])         Trainable=True
blocks.0.ln2.weight                                torch.Size([256])         Trainable=True
blocks.0.ln2.bias                                  torch.Size

# creating train.py 

In [48]:
%%writefile MiniGPT/train.py

import torch
from torch.utils.data import DataLoader

from src.config import MiniGPTConfig
from src.dataset import TinyStoriesDataset
from src.model import MiniGPT
from src.trainer import Trainer

# Config
config = MiniGPTConfig()

# Dataset
dataset = TinyStoriesDataset(config)

# DataLoader
train_loader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True
)

# Model
model = MiniGPT(config)

# Trainer
trainer = Trainer(
    model,
    train_loader,
    config
)

print("=" * 50)
print("Training Started...")
print("=" * 50)

for epoch in range(config.epochs):

    loss = trainer.train_epoch()

    print(
        f"Epoch {epoch+1}/{config.epochs} | Loss: {loss:.4f}"
    )

    # Save checkpoint after every epoch
    torch.save(
        model.state_dict(),
        f"MiniGPT/checkpoint_epoch_{epoch+1}.pth"
    )

    print(
        f"Checkpoint saved: MiniGPT/checkpoint_epoch_{epoch+1}.pth"
    )

print("Training Finished!")

# Save final model
torch.save(
    model.state_dict(),
    "MiniGPT/minigpt.pth"
)

print("Final model saved: MiniGPT/minigpt.pth")

Overwriting MiniGPT/train.py


## updated datset.py

In [50]:
%%writefile MiniGPT/src/dataset.py

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

from src.config import MiniGPTConfig


class TinyStoriesDataset(Dataset):

    def __init__(
        self,
        config: MiniGPTConfig,
        max_stories: int = 50000,
    ):

        self.config = config

        print("Loading tokenizer...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )

        print("Loading TinyStories...")

        dataset = load_dataset(
            "roneneldan/TinyStories",
            split="train"
        )

        dataset = dataset.select(
            range(max_stories)
        )

        print(f"Loaded {len(dataset)} stories")

        print("Tokenizing stories...")

        all_tokens = []

        for sample in dataset:

            tokens = self.tokenizer.encode(
                sample["text"],
                add_special_tokens=False
            )

            # Add EOS token after every story
            tokens.append(self.config.eos_token_id)

            all_tokens.extend(tokens)

        print(f"Total tokens: {len(all_tokens):,}")

        self.tokens = all_tokens

    def __len__(self):

        return (
            len(self.tokens) - 1
        ) // self.config.context_length

    def __getitem__(self, idx):

        start = idx * self.config.context_length
        end = start + self.config.context_length + 1

        chunk = self.tokens[start:end]

        input_ids = torch.tensor(
            chunk[:-1],
            dtype=torch.long
        )

        labels = torch.tensor(
            chunk[1:],
            dtype=torch.long
        )

        return input_ids, labels

Overwriting MiniGPT/src/dataset.py


In [51]:
import torch
from torch.utils.data import DataLoader

from src.config import MiniGPTConfig
from src.dataset import TinyStoriesDataset
from src.model import MiniGPT

print("=" * 70)
print("MiniGPT Debug")
print("=" * 70)

# Config
config = MiniGPTConfig()

# GPU Check
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

print()

# Dataset
dataset = TinyStoriesDataset(config)

print(f"Dataset Size   : {len(dataset)}")

# First Sample
input_ids, labels = dataset[0]

print(f"Input Shape    : {input_ids.shape}")
print(f"Label Shape    : {labels.shape}")

print()

# DataLoader
loader = DataLoader(
    dataset,
    batch_size=config.batch_size,
    shuffle=True
)

batch_inputs, batch_labels = next(iter(loader))

print(f"Batch Input    : {batch_inputs.shape}")
print(f"Batch Labels   : {batch_labels.shape}")

print()

# Model
model = MiniGPT(config)

output = model(batch_inputs)

print(f"Model Output   : {output.shape}")

print()

# Parameter Count
total = sum(p.numel() for p in model.parameters())

print(f"Parameters     : {total:,}")

print("=" * 70)

MiniGPT Debug
CUDA Available : False

Loading tokenizer...
Loading TinyStories...


Loaded 50000 stories
Tokenizing stories...
Total tokens: 10,496,407
Dataset Size   : 82003
Input Shape    : torch.Size([128])
Label Shape    : torch.Size([128])

Batch Input    : torch.Size([16, 128])
Batch Labels   : torch.Size([16, 128])

Model Output   : torch.Size([16, 128, 20000])

Parameters     : 8,309,248


In [52]:
import os

for file in os.listdir("/kaggle/working/MiniGPT/tokenizer"):
    print(file)

tokenizer.json
tokenizer_config.json


In [53]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("/kaggle/working/MiniGPT/tokenizer")

print(tok)
print(tok.special_tokens_map)

TokenizersBackend(name_or_path='/kaggle/working/MiniGPT/tokenizer', vocab_size=20000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<bos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)
{'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>'}


In [ ]:
!python MiniGPT/train.py

Loading tokenizer...
Loading TinyStories...
Loaded 50000 stories
Tokenizing stories...
Total tokens: 10,496,407
Training Started...
Training:   2%|▎             | 118/5126 [07:36<2:45:05,  1.98s/it, loss=20.2099]

In [ ]:
torch.save(model.state_dict(), "minigpt.pth")

# updated model.py 

In [54]:
%%writefile MiniGPT/src/model.py

import torch
import torch.nn as nn

from src.config import MiniGPTConfig
from src.embeddings import GPTEmbedding
from src.transformer import TransformerBlock


class MiniGPT(nn.Module):
    """
    MiniGPT Language Model
    """

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        # Save config for generate()
        self.config = config

        self.embedding = GPTEmbedding(config)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config)
                for _ in range(config.num_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(
            config.embedding_dim
        )

        self.lm_head = nn.Linear(
            config.embedding_dim,
            config.vocab_size,
            bias=False
        )

        # Weight Tying (GPT-2 Style)
        self.lm_head.weight = self.embedding.token_embedding.weight

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50, # this is for choose only the top50 best token in all the possiblility. ###Experimental 
    ):
        """
        Greedy text generation.
        """

        self.eval()

        for _ in range(max_new_tokens):

            # Keep only the last context_length tokens
            input_crop = input_ids[:, -self.config.context_length:]

            # Forward pass
            logits = self(input_crop)

            # Use only the last token prediction
            logits = logits[:, -1, :]

            #temprature
            #temperature = 0.8                  ###experimental

            logits = logits / temperature


            # Top K filtering ####this is the whole experimental
            values, indices = torch.topk(
                logits,
                top_k
            )

            filtered_logits = torch.full_like(
                logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                dim=-1,
                index=indices,
                src=values
            )
            

            # Convert logits to probabilities
            probs = torch.softmax(
                logits,
                dim=-1
            )

            # Convert logits to probabilities
            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            # Append predicted token
            input_ids = torch.cat(
                (input_ids, next_token),
                dim=1
            )

        return input_ids

Overwriting MiniGPT/src/model.py


# adding top_k and top_p for the model generated the response 

In [59]:
%%writefile MiniGPT/src/model.py

import torch
import torch.nn as nn

from src.config import MiniGPTConfig
from src.embeddings import GPTEmbedding
from src.transformer import TransformerBlock


class MiniGPT(nn.Module):
    """
    MiniGPT Language Model
    """

    def __init__(self, config: MiniGPTConfig):
        super().__init__()

        # Save config for generate()
        self.config = config

        self.embedding = GPTEmbedding(config)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config)
                for _ in range(config.num_layers)
            ]
        )

        self.final_norm = nn.LayerNorm(
            config.embedding_dim
        )

        self.lm_head = nn.Linear(
            config.embedding_dim,
            config.vocab_size,
            bias=False
        )

        # Weight Tying (GPT-2 Style)
        self.lm_head.weight = self.embedding.token_embedding.weight

    def forward(self, input_ids):
        x = self.embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens=50,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    ):
        """
        Generate text using Temperature + Top-k + Top-p (Nucleus) Sampling.
        """

        self.eval()

        for _ in range(max_new_tokens):

            # Keep only the last context window
            input_crop = input_ids[:, -self.config.context_length:]

            # Forward pass
            logits = self(input_crop)

            # Take logits for the last token only
            logits = logits[:, -1, :]

            # -------------------------
            # Temperature
            # -------------------------
            logits = logits / temperature

            # -------------------------
            # Top-k
            # -------------------------
            if top_k is not None:

                values, indices = torch.topk(
                    logits,
                    k=min(top_k, logits.size(-1))
                )

                filtered_logits = torch.full_like(
                    logits,
                    float("-inf")
                )

                filtered_logits.scatter_(
                    dim=-1,
                    index=indices,
                    src=values
                )

                logits = filtered_logits

            # -------------------------
            # Convert to probabilities
            # -------------------------
            probs = torch.softmax(
                logits,
                dim=-1
            )

            # -------------------------
            # Top-p (Nucleus Sampling)
            # -------------------------
            if top_p is not None:

                sorted_probs, sorted_indices = torch.sort(
                    probs,
                    descending=True
                )

                cumulative_probs = torch.cumsum(
                    sorted_probs,
                    dim=-1
                )

                sorted_indices_to_remove = cumulative_probs > top_p

                # Always keep at least one token
                sorted_indices_to_remove[..., 1:] = \
                    sorted_indices_to_remove[..., :-1].clone()

                sorted_indices_to_remove[..., 0] = False

                sorted_probs[sorted_indices_to_remove] = 0

                # Renormalize
                sorted_probs = sorted_probs / sorted_probs.sum(
                    dim=-1,
                    keepdim=True
                )

                sampled = torch.multinomial(
                    sorted_probs,
                    num_samples=1
                )

                next_token = sorted_indices.gather(
                    dim=-1,
                    index=sampled
                )
                
                # Stop if EOS token is generated
                if next_token.item() == self.config.eos_token_id:
                    break

            else:

                next_token = torch.multinomial(
                    probs,
                    num_samples=1
                )

            # Append token
            input_ids = torch.cat(
                (input_ids, next_token),
                dim=1
            )

        return input_ids

Overwriting MiniGPT/src/model.py


In [56]:
print("Training Finished!")

# Save trained model
torch.save(
    model.state_dict(),
    "MiniGPT/minigpt.pth"
)

print("Model saved to MiniGPT/minigpt.pth")

Training Finished!
Model saved to MiniGPT/minigpt.pth


# test our model 

In [57]:
%%writefile MiniGPT/generate.py

import torch
from transformers import AutoTokenizer

from src.config import MiniGPTConfig
from src.model import MiniGPT


# -----------------------
# Configuration
# -----------------------

config = MiniGPTConfig()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# -----------------------
# Load Tokenizer
# -----------------------

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    config.tokenizer_path
)

# -----------------------
# Load Model
# -----------------------

print("Loading model...")

model = MiniGPT(config)

model.load_state_dict(
    torch.load(
        "/kaggle/input/models/sameerdhara/minigpt/transformers/version-0.0001/1/minigpt.pth",  ##MiniGPT/minigpt.pth 1st path 
        map_location=device
    )
)

model.to(device)

model.eval()

# -----------------------
# Prompt
# -----------------------

prompt = "There was a little girl with dark hair  "

print(f"\nPrompt:\n{prompt}\n")

input_ids = tokenizer.encode(
    prompt,
    return_tensors="pt"
).to(device)

# -----------------------
# Generate
# -----------------------

output = model.generate(
    input_ids,
    max_new_tokens=100,
    temperature=0.5, # this temperature  i found the best overall 
    top_k=100,
    top_p=0.9,
)

# -----------------------
# Decode
# -----------------------

text = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("=" * 80)
print(text)
print("=" * 80)

Writing MiniGPT/generate.py


In [58]:
!python MiniGPT/generate.py

Loading tokenizer...
Loading model...
Traceback (most recent call last):
  File "/kaggle/working/MiniGPT/generate.py", line 37, in <module>
    model.load_state_dict(
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 2635, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for MiniGPT:
	size mismatch for embedding.token_embedding.weight: copying a param with shape torch.Size([8000, 256]) from checkpoint, the shape in current model is torch.Size([20000, 256]).
	size mismatch for lm_head.weight: copying a param with shape torch.Size([8000, 256]) from checkpoint, the shape in current model is torch.Size([20000, 256]).


In [ ]:
from IPython.display import FileLink

FileLink("MiniGPT/minigpt.pth")